In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from tqdm import tqdm
from alignment.prompts import OEQA_INSTRUCTION_TEMPLATE, OEQA_SYSTEM_PROMPT, MCQA_SYSTEM_PROMPT, MCQA_INSTRUCTION_TEMPLATE

# # Qwen3-14B
# tokenizer = AutoTokenizer.from_pretrained("/gpfs/home/eungizoa/models/doctrio/qwen3-14b-mcqa-oeqa")
# model = AutoModelForCausalLM.from_pretrained("/gpfs/home/eungizoa/models/doctrio/qwen3-14b-mcqa-oeqa", torch_dtype=torch.bfloat16).to("cuda:0")

# raw_datasets = pd.read_csv("/gpfs/home/eungizoa/data/hydra-fineval/preliminary/test.parsed.csv").to_dict(orient="records")
# for example in raw_datasets:
#     example["options"] = eval(example["options"])
#     if example["options"]:
#         system_message = MCQA_SYSTEM_PROMPT
#         options_str = "\n".join([f"{i+1}. {option}" for i, option in enumerate(example["options"])])
#         user_message = MCQA_INSTRUCTION_TEMPLATE.format(query=example["query"], options=options_str)
#     else:
#         system_message = OEQA_SYSTEM_PROMPT
#         user_message = OEQA_INSTRUCTION_TEMPLATE.format(query=example["query"])
        
#     example["messages"] = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

# # Exaone-3.5B
tokenizer = AutoTokenizer.from_pretrained("/gpfs/home/eungizoa/models/doctrio/exaone3.5-7.5b-mcqa-oeqa", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("/gpfs/home/eungizoa/models/doctrio/exaone3.5-7.5b-mcqa-oeqa", torch_dtype=torch.bfloat16, trust_remote_code=True).to("cuda:0")

raw_datasets = pd.read_csv("/gpfs/home/eungizoa/data/hydra-fineval/preliminary/test.parsed.csv").to_dict(orient="records")
for example in raw_datasets:
    example["options"] = eval(example["options"])
    if example["options"]:
        system_message = MCQA_SYSTEM_PROMPT
        options_str = "\n".join([f"{i+1}. {option}" for i, option in enumerate(example["options"])])
        user_message = MCQA_INSTRUCTION_TEMPLATE.format(query=example["query"], options=options_str)
    else:
        system_message = OEQA_SYSTEM_PROMPT
        user_message = OEQA_INSTRUCTION_TEMPLATE.format(query=example["query"])
        
    example["messages"] = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
# # Qwen3-14B
# model.eval()

# with torch.no_grad():
    
#     for example in tqdm(raw_datasets):
#         model_inputs = tokenizer.apply_chat_template(example["messages"], tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors="pt", return_dict=True)
#         outputs = model.generate(**model_inputs.to(model.device), max_new_tokens=2048)
#         response_ids = outputs[0][len(model_inputs["input_ids"][0]):].tolist()
#         response = tokenizer.decode(response_ids, skip_special_tokens=True)
#         example["response"] = response

# Exaone-3.5B
model.eval()

with torch.no_grad():
    
    for example in tqdm(raw_datasets):
        model_inputs = tokenizer.apply_chat_template(example["messages"], tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        outputs = model.generate(**model_inputs.to(model.device), max_new_tokens=2048)
        response_ids = outputs[0][len(model_inputs["input_ids"][0]):].tolist()
        response = tokenizer.decode(response_ids, skip_special_tokens=True)
        example["response"] = response

In [ ]:
pd.DataFrame(raw_datasets).iloc[:, 1:].to_csv("/gpfs/home/eungizoa/data/hydra-fineval/submission/exaone-250825-16pm.csv", index=None, encoding="utf-8-sig")

In [ ]:
import os
import re

def extract_cot_and_final_answer(text):
    """
    Extract thinking traces and final answer from text.
    
    Returns:
        tuple: (thinking_trace, answer)
    """
    # Pattern to match thinking traces (everything before "Answer:")
    thinking_pattern = r'^(.*?)(?=Answer:\s*)'
    
    # Pattern to match the answer after "Answer:"
    answer_pattern = r'.*Answer:\s*(.*)$'
    
    thinking_match = re.search(thinking_pattern, text, re.DOTALL | re.MULTILINE)
    answer_match = list(re.finditer(answer_pattern, text, re.DOTALL | re.MULTILINE))
    if answer_match:
        answer_match = answer_match[-1]
    else:
        return "", ""
    
    thinking_trace = thinking_match.group(1).strip() if thinking_match else ""
    answer = answer_match.group(1).strip() if answer_match else ""
    
    return thinking_trace.strip(), answer.strip()

for example in raw_datasets:
    cot_trace, answer = extract_cot_and_final_answer(example["response"])
    example["is_mcqa"] = True if example["options"] else False
    if example["is_mcqa"]:
        example["parsed_cot_trace"] = cot_trace
        example["parsed_answer"] = answer if answer else "0"
    else:
        example["parsed_cot_trace"] = cot_trace
        example["parsed_answer"] = answer

In [ ]:
submission = pd.DataFrame([{"ID": example["ID"], "Answer": example["parsed_answer"], "is_mcqa": example["is_mcqa"]} for example in raw_datasets])

In [ ]:
submission[submission["is_mcqa"] == True]["Answer"].value_counts()

In [ ]:
submission[(submission["is_mcqa"] == True) & (submission["Answer"] == "0")]

In [ ]:
[example for example in raw_datasets if example["ID"] == "TEST_143"][0]

In [ ]:
submission.drop(columns=["is_mcqa"]).to_csv("/gpfs/home/eungizoa/data/hydra-fineval/submission/submission-exaone-250825-16pm.csv", encoding="utf-8-sig", index=None)